<div >
<img src = "../banner.jpg" />
</div>

<a href="https://colab.research.google.com/github/ignaciomsarmiento/BDML_202501/blob/main/Modulo07/01_CuadernoModulo07_SpatialData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial Data

This notebook accompanies the lecture on **spatial data and spatial cross-validation**. The central argument, which the slides develop algebraically, is:

> When observations are spatially autocorrelated, standard random cross-validation **underestimates** out-of-sample error — because validation points are close to training points and share the same spatially correlated noise. Spatial cross-validation corrects this by enforcing geographic separation between folds.

We build this argument in three steps:
1. Load a spatial dataset and visualise the autocorrelation structure
2. Compare three spatial CV strategies (blocks, leave-location-out, buffers)
3. Demonstrate the bias concretely: first with an Elastic Net, then with a Deep Neural Network — where the bias is amplified by the model's greater capacity to overfit

In [ ]:
require("pacman")
p_load("tidyverse","sf","modeldata", "caret","keras3")

In [ ]:
data("ames", package = "modeldata")

In [ ]:
head(ames)

In [ ]:
dim(ames)

In [ ]:
class(ames)

![](figs/mercator.gif)

In [ ]:
#For speed, I'm going to keep the ten neighborhoods with the most observations, 
ames <- ames  %>% filter(Neighborhood %in%c("North_Ames", "College_Creek", 
                                          "Old_Town", "Edwards", "Somerset", 
                                          "Northridge_Heights", "Gilbert", "Sawyer", 
                                         "Northwest_Ames", "Sawyer_West"))
ames$Neighborhood <- droplevels(ames$Neighborhood)

In [ ]:
table(ames$Neighborhood)

In [ ]:
                                     
ames_sf <- sf::st_as_sf(
  ames,
  # "coords" is in x/y order -- so longitude goes first!
  coords = c( "Longitude","Latitude"),
  remove=FALSE,
  # Set our coordinate reference system to EPSG:4326,
  # the standard WGS84 geodetic coordinate reference system
  crs = 4326
)

In [ ]:
#?st_as_sf

In [ ]:
class(ames_sf)

In [ ]:
head(data.frame(ames_sf))

In [ ]:
#graficar con ggplot
ggplot() +
    geom_sf(data=ames_sf)+
    theme_bw()

In [ ]:
p_load("leaflet")

In [ ]:


my_colors <- colorRampPalette(RColorBrewer::brewer.pal(9, "Set1"))(10)
pal <- colorFactor(palette = my_colors, domain = ames_sf$Neighborhood)

p_load("htmlwidgets") # para verlo bien en jupyter notebook, no es necesario en otras plataformas

m <- leaflet(ames_sf, width = "100%", height = 700) %>%
  addProviderTiles("CartoDB.Positron") %>%
  addCircleMarkers(
    color       = ~pal(Neighborhood),
    fillColor   = ~pal(Neighborhood),
    fillOpacity = 0.7,
    radius      = 4,
    stroke      = FALSE,
    label       = ~Neighborhood
  ) %>%
  addLegend(pal = pal, values = ~Neighborhood, title = "Neighborhood")

saveWidget(m, "map.html", selfcontained = TRUE)


In [ ]:
html_content <- readLines("map.html") %>% paste(collapse = "\n")
IRdisplay::display_html(
  paste0('<div style="width:100%; height:700px; overflow:hidden;">', 
         html_content, 
         '</div>')
)

## Spatial Autocorrelation

This relationship may exhibit spatial autocorrelation across the city of Ames, and we can investigate it using any of the several methods provided by the spatialsample. 


### Why does this matter for cross-validation?

Recall from the slides if $\text{Cov}(\varepsilon(s_i),\, \varepsilon(s_j)) > 0$ for nearby locations, a model trained on $s_j$ can memorise the local noise $\varepsilon(s_j)$. In a random CV, the validation point $s_i$ is likely to be geographically close to some training point, so its residual is correlated with what the model learned, making the validation error look artificially small.

The three strategies below all address this by ensuring **geographic separation** between training and validation observations within each fold.

In [ ]:
p_load("spatialsample")

### Spatial Blocks

For instance, the `spatial_block_cv()` function will perform [spatial blocking](https://doi.org/10.1111/ecog.02881) with your data:

In [ ]:
set.seed(123)
block_folds <- spatial_block_cv(ames_sf, v = 5)

Here, the seed ensures that the sampling results are reproducible. Then, `spatial_block_cv` divides the spatial dataset `ames_sf` into 5 folds for cross-validation, ensuring the training and testing sets are spatially separated. This prevents information from geographically close observations from leaking between folds. 

`Autoplot` will give us a clear visual of how the data was split into blocks, helping us understand the spatial structure of the validation scheme.

In [ ]:
p_load("purrr")

walk(block_folds$splits, function(x) print(autoplot(x)))

### Spatial LLOCV

If you already have a sense of what locations in your data are likely to be closely related, you can also use the `spatial_leave_location_out_cv()` function to perform [leave-location-out cross-validation](https://doi.org/10.1016/j.envsoft.2017.12.001). 

For instance, we can split the Ames data into folds based on neighborhoods using this function:

In [ ]:
set.seed(123)

location_folds <- 
  spatial_leave_location_out_cv(
    ames_sf,
    group = Neighborhood
  )

In [ ]:
p_load("purrr")

walk(location_folds$splits, function(x) print(autoplot(x)))

### Spatial Buffers

The `spatial_buffer_vfold_cv()` function will perform [spatially buffered cross-validation](https://onlinelibrary.wiley.com/doi/10.1111/geb.12161) with your data:

In [ ]:
set.seed(123)

buffer_folds <- spatial_buffer_vfold_cv(ames_sf, radius=200,buffer=200)

In [ ]:
autoplot(buffer_folds$splits[[3]]) + theme_bw()

 Here we use a 200-meter radius and a 200-meter buffer. Although the input data is in degrees (EPSG:4326), `spatial_buffer_vfold_cv()` appears to perform a **sanity check**: since 200 degrees would represent an enormous distance (over half the Earth's circumference, which is approximately 40,075 km), the function internally treats the values as if they are in **meters** rather than blindly applying them as degrees. This behavior likely prevents "weird" results when users provide realistic buffer sizes but forget to project the data. 

Nonetheless, it's **best practice** to explicitly transform your data to a projected CRS in meters (e.g., UTM) to avoid relying on implicit assumptions and ensure spatial distances are handled consistently.



In [ ]:
ames_sf2 <- st_transform(ames_sf, crs = 26915)  # UTM zone 15N, in meters   https://spatialreference.org/ref/epsg/26915/

set.seed(123)
buffer_folds2 <- spatial_buffer_vfold_cv(ames_sf2, radius = 200, buffer = 200)

In [ ]:
autoplot(buffer_folds2$splits[[3]]) + theme_bw()

## Example following the Problem Set

We now simulate a scenario close to the problem set.

**Setup:**
- **Test set** → `North_Ames` (held out entirely: a neighborhood the model never sees during training, like Chapineron in the Problem Set)
- **Training set** → all other 9 neighborhoods

We fit two Elastic Net models, 


$$\begin{aligned}
\min_{\beta}\; \mathrm{EN}(\beta) = \sum_{i=1}^n \left(y_i - \beta_0 - \sum_{j=1}^p x_{ij}\beta_j\right)^2 + \lambda\!\left(\alpha \sum_{j=1}^p |\beta_j| + \frac{1-\alpha}{2} \sum_{j=1}^p \beta_j^2\right)
\end{aligned}$$


identical in everything except how cross-validation is done. We tune $(\lambda, \alpha)$ separately under each CV scheme. The key question: **Which one predicts best out of sample?**

| Model | CV strategy | What the folds look like |
|-------|-------------|---------------------------|
| `EN_tp_random` | Standard 5-fold random CV | Observations randomly shuffled into folds. Neighboring houses can appear in both train and validation |
| `EN_tp_spatial` | Leave-one-neighborhood-out | Each fold withholds a full neighborhood. Clean geographic separation |



We start by separating the testing data set (the one tha would be in Kaggle, our Chapinero), and the trainig data

In [ ]:
test  <- ames_sf  %>% filter(Neighborhood=="North_Ames")

train <- ames_sf  %>% filter(Neighborhood!="North_Ames")

## Specification

- Gr_Liv_Area: Above grade (ground) living area square feet
- Lot_Frontage: Linear feet of street connected to property
- Lot_Area: Lot size in square feet
- First_Flr_SF: First Floor square feet
- Garage_Area: Size of garage in square feet
- Sale_Price: Sale price $$


In [ ]:
## Specification

fml_en <- formula(log(Sale_Price) ~  Gr_Liv_Area + Lot_Frontage + Lot_Area + First_Flr_SF + Garage_Area)

### Train it with random CV

In [ ]:
fitControl_tp_random<-trainControl(method ="cv",
                         number=5)


In [ ]:
set.seed(123)

EN_tp_random<-train(fml_en,
             data=train,
             method = 'glmnet', 
             trControl = fitControl_tp_random,
             metric="MAE",
             tuneGrid = expand.grid(alpha =seq(0,1,length.out = 10),
                                    lambda = seq(0.001,0.2,length.out = 10))
              ) 

### Train it with spatial CV

In [ ]:
set.seed(123)

location_folds_train <- 
  spatial_leave_location_out_cv(
    train,
    group = Neighborhood
  )


In [ ]:
autoplot(location_folds_train)

We need to extract the indices to fit the folds

In [ ]:
folds_train<-list()

for(i in 1:length(location_folds_train$splits)){
  folds_train[[i]]<- location_folds_train$splits[[i]]$in_id
}

This code extracts the training indices from each spatial fold and stores them in a list that caret can handle directly. `location_folds_train` is a spatial cross-validation object in which each split knows which observations belong to the training set ($in_id) and which to the validation set. The loop simply unpacks that information: for each fold i, it pulls out the row indices of the training observations and stores them in `folds_train[[i]]`.

The key here is that I use this object then to inex my folds, with this I turn a standard elastic net into a spatially cross-validated one.

In [ ]:

fitControl_spatial<-trainControl(method ="cv",
                         index=folds_train)

In [ ]:
set.seed(123)

EN_tp_spatial<-train(fml_en,
             data=train,
             method = 'glmnet', 
             trControl = fitControl_spatial,
             metric="MAE",
             tuneGrid = expand.grid(alpha =seq(0,1,length.out = 10),
                                    lambda = seq(0.001,0.2,length.out = 10))
              ) 

In [ ]:
#EN_tp_random

In [ ]:
#EN_tp$bestTune

## What would be our Kaggle's score?

### Predict out of sample

In [ ]:
test$log_price_hat_random<-predict(EN_tp_random,newdata = test)

In [ ]:
test$log_price_hat_spatial<-predict(EN_tp_spatial,newdata = test)

In [ ]:
test<- test  %>% mutate(price_hat_random=exp(log_price_hat_random),price_hat_spatial=exp(log_price_hat_spatial))

Calculate the MAE

In [ ]:
#MAE
mean(abs(test$Sale_Price-test$price_hat_random))

In [ ]:
mean(abs(test$Sale_Price-test$price_hat_spatial))

Tip: Prices are rounded, simple correction

In [ ]:
mean(abs(test$Sale_Price-floor(test$price_hat_spatial)))

## Deep Feedforward Network with Dropout

The Elastic Net is well-regularised and low-capacity. What happens with a **deep neural network**, many more parameters, on the *same small dataset*?

Neural networks, especially deep ones, can memorize local noise patterns more aggressively, so CV becomes *more* optimistic.



### Network preparation

Keep the same variables, and drop the geometry 

In [ ]:
nn_vars <- c( "Gr_Liv_Area", "Lot_Frontage", "Lot_Area", "First_Flr_SF", "Garage_Area")
 

# Drop sf geometry, select variables, remove Nas
train_nn <- train %>%
  st_drop_geometry() %>%
  select(Sale_Price, Neighborhood, all_of(nn_vars)) 

test_nn <- test %>%
  st_drop_geometry() %>%
  select(Sale_Price, Neighborhood, all_of(nn_vars)) 


Turn into a model matrix, and build a validation neighborhood: North Ames. The idea is that for homework you build the entire loop (it can be quite challenging)

In [ ]:
# model.matrix: "one hot encoding"

# Testing data set North_Ames
x_test_nn  <- model.matrix(~ . - Sale_Price - Neighborhood - 1, data = test_nn)
y_test_nn  <- log(test_nn$Sale_Price)


# Training data set, the other 9 neighborhoods
x_train_nn <- model.matrix(~ . - Sale_Price - Neighborhood - 1, data = train_nn)
y_train_nn <- log(train_nn$Sale_Price)


In [ ]:
# Validation set Northwest_Ames (very close) may lead to overfit
validation_neighborhood <- "Northwest_Ames"

validation <- train_nn  %>% filter(Neighborhood %in% validation_neighborhood) 
train_wv   <- train_nn  %>% filter(!(Neighborhood %in% validation_neighborhood))

# validation set matrices
x_val <- model.matrix(~ . - Sale_Price - Neighborhood - 1, data = validation)
y_val <- log(validation$Sale_Price)       

# training set matrices
x_train_wv <- model.matrix(~ . - Sale_Price - Neighborhood - 1, data = train_wv)
y_train_wv <- log(train_wv$Sale_Price)    

In [ ]:
mu    <- colMeans(x_train_nn)
sigma <- apply(x_train_nn, 2, sd)
sigma[sigma < 1e-8] <- 1

x_train_nn <- scale(x_train_nn, center = mu, scale = sigma)
x_train_wv <- scale(x_train_wv, center = mu, scale = sigma)
x_val      <- scale(x_val,      center = mu, scale = sigma)
x_test_nn  <- scale(x_test_nn,  center = mu, scale = sigma)

We use the training set statistics (mean and standard deviation) to scale **all** splits: training, validation, and test, for the same reason we never peek at the test set when selecting hyperparameters: avoiding data leakage.

In practice, when you train a model you have no access to future observations. If you computed the mean and standard deviation from the validation or test set, you would be using information the model is not supposed to see.

The rule is: **scaling is defined on the training set and applied everywhere**. Think of it as saying "the world is centered on the average of what I saw during training." When a new observation arrives at test time, you scale it using those same training parameters, not its own. A house in the test set with `Lot_Area = 13,000` should be transformed as `(13,000 − μ_train) / σ_train`, not re-centered on the test set's own mean otherwise the value it receives is inconsistent with what the network learned.

### Model Architecture


In [ ]:
set.seed(1234) # reproducibilidad

model_nn <- keras_model_sequential(input_shape = ncol(x_train_wv)) %>%
    layer_dense(units = 64,  activation = "relu") %>%
    layer_dropout(rate = 0.3) %>%
    layer_dense(units = 32,  activation = "relu") %>%
    layer_dropout(rate = 0.2) %>%
    layer_dense(units = 1)

model_nn %>% compile(
    optimizer = optimizer_adam(learning_rate = 0.001),
    loss      = "mse",
    metrics   = "mae"
  )

model_nn


### Northwest_Ames as spatial crossvalidation

In [ ]:
set.seed(1234) # reproducibilidad

history <- model_nn %>% fit(
  x_train_wv, y_train_wv,
  epochs          = 60,
  batch_size      = 16,
  validation_data = list(x_val, y_val),   
  callbacks       = list(
    callback_early_stopping(
      monitor              = "val_loss",
      patience             = 10,
      restore_best_weights = TRUE
    )
  ),
  verbose = 1
)

In [ ]:
plot(history )

In [ ]:
eval_spatial_validation <- model_nn %>% evaluate(x_val, y_val)
eval_spatial_validation

In [ ]:
test$price_hat_spatial_nn <- exp(as.vector(model_nn %>% predict(x_test_nn, verbose = 0)))

mean(abs(test$Sale_Price-test$price_hat_spatial_nn))

Compare to our best EN model

In [ ]:
mean(abs(test$Sale_Price-floor(test$price_hat_spatial)))

This is **one fold** of a leave-one-neighborhood-out spatial CV. We trained on 8 neighborhoods, withheld `Northwest_Ames`, and measured how well the model generalizes to that held-out area.

To get a proper CV estimate, we need to repeat this for every neighborhood and average the MAEs. **Homework:** come up with a function that extends this function to do spatial cross-validation

### Random Validation Neural Network

Now the validation set ignores the geography

In [ ]:
set.seed(1234) # reproducibilidad

model_nn_random <- keras_model_sequential(input_shape = ncol(x_train_wv)) %>%
    layer_dense(units = 64,  activation = "relu") %>%
    layer_dropout(rate = 0.3) %>%
    layer_dense(units = 32,  activation = "relu") %>%
    layer_dropout(rate = 0.2) %>%
    layer_dense(units = 1)

model_nn_random %>% compile(
    optimizer = optimizer_adam(learning_rate = 0.001),
    loss      = "mse",
    metrics   = "mae"
  )

model_nn_random


In [ ]:
set.seed(1234) # reproducibilidad

history_val <- model_nn_random %>% fit(
  x_train_nn, y_train_nn,
  epochs          = 60,
  batch_size      = 16,
   validation_split = 0.2,   
  callbacks       = list(
    callback_early_stopping(
      monitor              = "val_loss",
      patience             = 10,
      restore_best_weights = TRUE
    )
  ),
  verbose = 1
)


In [ ]:
plot(history_val)

In [ ]:
eval_random_validation <- model_nn_random %>% evaluate( x_train_nn, y_train_nn)
# In validation Sample
eval_random_validation

In [ ]:
# Out of Sample
test$price_hat_random_nn <- exp(as.vector(model_nn_random %>% predict(x_test_nn, verbose = 0)))
mean(abs(test$Sale_Price-test$price_hat_random_nn))

Compared to the spatial model 

In [ ]:
# In validation Sample
eval_spatial_validation

In [ ]:
# Out of Sample
mean(abs(test$Sale_Price-test$price_hat_spatial_nn))

Compare to our best EN model

In [ ]:
mean(abs(test$Sale_Price-floor(test$price_hat_spatial)))

In [ ]:
quantile(test$price_hat_random_nn, c(0.25, 0.5, 0.75, 0.95, 1))

In [ ]:
test_nn$price_hat_random_nn <- exp(as.vector(model_nn_random %>% predict(x_test_nn, verbose = 0)))
View(test_nn %>% filter(test$price_hat_random_nn>quantile(test_nn$price_hat_random_nn,.99)))

In [ ]:
summary(train_nn)

North Ames contains houses that are larger and more expensive than the typical house in the nine training neighborhoods. The network learned the relationship between features and log(price) only within the range it saw during training. When it encounters a house with `Gr_Liv_Area` or `Garage_Area` outside that range, the dense layers extrapolate without bound, pushing the predicted log(price) to values like 20, which after `exp()` becomes hundreds of millions of dollars.

The Elastic Net does not have this problem because the penalty shrinks all coefficients toward zero. When a new observation arrives with extreme feature values, the prediction is pulled back toward the mean rather than allowed to grow without limit. Regularization acts as a natural brake on extrapolation.

This is the deeper lesson: the problem is not just spatial autocorrelation. Random cross-validation never exposes the model to a geographically new neighborhood during training, so it has no way to detect that it will fail when it does.